# Cost-Benefit Score (Phase 1 baseline)

Objective: compute a cost-benefit score combining player performance
(2024 season) and market value, as a traditional efficiency-based scouting
baseline — the type of system this project's underlying thesis critiques,
built here as a technical foundation (see README).

## Known limitation: temporal mismatch between value and performance

Market value data (Transfermarkt) reflects the most recent valuation
available, not the player's value specifically in 2024 — while performance
stats (Eduardo Palmieri) are specifically from the 2024 season. This means
the cost-benefit score answers "is this player's current valuation
consistent with their 2024 performance?" rather than "was this player
underpriced relative to their 2024 performance at the time."

In [29]:
import pandas as pd

## Loading cleaned data

In [30]:
df = pd.read_csv('../data/processed/players_final_2024.csv')

## Filtering to players with matched stats

In [31]:
df_scored = df[df['stats_player_name'].notna()]
df_scored.shape

(326, 39)

## Grouping players by position

In [32]:
df_scored['stats_position'].unique()

<StringArray>
[   'GK',    'LM',    'CB',    'AM',    'RB',    'CM',    'RM', 'FW,AM',
 'AM,DM',    'RW',    'FW',    'LB',    'WB',    'LW', 'RM,FW', 'FW,LW',
    'DM', 'CB,CM', 'RM,LM', 'FW,LM', 'WB,CM', 'LM,RM', 'DM,CB', 'RW,LW',
 'CM,RM', 'FW,RW', 'LB,CM', 'CB,RW', 'LM,DM', 'LM,FW', 'RB,CB', 'RM,LW']
Length: 32, dtype: str

In [33]:
df_scored['sub_position'].unique()

<StringArray>
[        'Goalkeeper', 'Attacking Midfield',        'Centre-Back',
         'Right-Back',        'Left Winger',   'Central Midfield',
     'Centre-Forward',       'Right Winger',          'Left-Back',
 'Defensive Midfield',                  nan]
Length: 11, dtype: str

In [34]:
position_groups = {
    'Goalkeeper': ['Goalkeeper'],
    'Centre-Back': ['Centre-Back'],
    'Full-Back': ['Right-Back', 'Left-Back'],
    'Defensive Midfielder': ['Defensive Midfield'],
    'Central Midfielder': ['Central Midfield'],
    'Attacking Midfielder': ['Attacking Midfield'],
    'Forward': ['Right Winger', 'Left Winger', 'Centre-Forward'],
}

def map_position_group(position):
    for group, positions in position_groups.items():
        if position in positions:
            return group
    return 'Unknown'

df_scored['position_group'] = df_scored['sub_position'].apply(map_position_group)
df_scored.loc[df_scored['player_name'] == 'Pedro Rangel', 'position_group'] = 'Goalkeeper'
df_scored['position_group'].value_counts()

position_group
Forward                 85
Centre-Back             62
Full-Back               50
Defensive Midfielder    41
Attacking Midfielder    30
Goalkeeper              29
Central Midfielder      29
Name: count, dtype: int64

## Normalizing metrics per 90 minutes

In [35]:
df_scored_filtered = df_scored[df_scored['minutes_played'] >= 900]
df_scored_filtered.shape

(198, 40)

In [36]:
metrics_to_normalize = ['goals', 'assists', 'xG', 'xAG', 'progressive_passes',
                         'progressive_carries', 'dribbles_successful', 'shot_creating_actions',
                         'goal_creating_actions', 'tackles', 'blocks']

for metric in metrics_to_normalize:
    df_scored_filtered[f'{metric}_per90'] = (df_scored_filtered[metric] / df_scored_filtered['minutes_played']) * 90

df_scored_filtered[[f'{m}_per90' for m in metrics_to_normalize]].head()

,goals_per90,assists_per90,xG_per90,xAG_per90,progressive_passes_per90,progressive_carries_per90,dribbles_successful_per90,shot_creating_actions_per90,goal_creating_actions_per90,tackles_per90,blocks_per90
0,0.000000,0.000000,0.007673,0.005115,0.332481,0.281330,0.076726,0.358056,0.025575,0.204604,0.102302
1,0.179283,0.239044,0.095618,0.256972,4.362550,1.374502,0.717131,4.482072,0.239044,0.597610,0.478088
2,0.000000,0.000000,0.035714,0.050000,2.428571,0.142857,0.000000,0.928571,0.000000,0.928571,0.714286
3,0.221311,0.221311,0.125410,0.184426,7.450820,1.770492,1.254098,4.500000,0.663934,0.885246,0.147541
4,0.075885,0.000000,0.022766,0.045531,2.428331,0.151771,0.075885,0.682968,0.151771,1.290051,0.531197


## Excluding goalkeepers (no suitable metrics available)

### Note: goalkeepers excluded from scoring

None of the available performance metrics (goals, assists, progressive
actions, outfield defensive actions) meaningfully evaluate goalkeeper
performance. Proper goalkeeper analysis would require metrics like saves,
save percentage, or goals prevented — not present in this dataset.
Goalkeepers are excluded from the cost-benefit score in this version;
adding them is a documented next step, pending a suitable data source.

In [37]:
df_outfield = df_scored_filtered[df_scored_filtered['position_group'] != 'Goalkeeper']
df_outfield['position_group'].value_counts()

position_group
Forward                 47
Centre-Back             41
Full-Back               35
Attacking Midfielder    21
Defensive Midfielder    21
Central Midfielder      17
Name: count, dtype: int64

## Defining position-specific weights

In [38]:
position_weights = {
    'Forward': {
        'goals_per90': 2.5,
        'xG_per90': 3.0,
        'assists_per90': 1.5,
        'xAG_per90': 1.5,
        'shot_creating_actions_per90': 1.0,
        'goal_creating_actions_per90': 1.5,
        'dribbles_successful_per90': 1.0,
    },
    'Attacking Midfielder': {
        'xAG_per90': 3.0,
        'assists_per90': 2.0,
        'shot_creating_actions_per90': 2.5,
        'goal_creating_actions_per90': 2.0,
        'goals_per90': 1.5,
        'xG_per90': 1.5,
        'dribbles_successful_per90': 1.0,
    },
    'Central Midfielder': {
        'progressive_passes_per90': 3.0,
        'progressive_carries_per90': 1.5,
        'xAG_per90': 2.0,
        'assists_per90': 1.0,
        'shot_creating_actions_per90': 1.5,
        'tackles_per90': 1.5,
        'blocks_per90': 1.0,
    },
    'Defensive Midfielder': {
        'tackles_per90': 2.0,
        'blocks_per90': 1.5,
        'progressive_passes_per90': 3.0,
        'progressive_carries_per90': 1.0,
        'xAG_per90': 0.75,
        'assists_per90': 0.25,
    },
    'Full-Back': {
        'progressive_carries_per90': 2.5,
        'progressive_passes_per90': 2.5,
        'assists_per90': 1.5,
        'xAG_per90': 2.0,
        'tackles_per90': 1.5,
        'blocks_per90': 1.5,
        'dribbles_successful_per90': 1.0,
    },
    'Centre-Back': {
        'tackles_per90': 2.0,
        'blocks_per90': 2.0,
        'progressive_passes_per90': 2.5,
        'progressive_carries_per90': 1.0,
    },
}

## Calculating weighted performance score

In [39]:
def calculate_weighted_score(row):
    weights = position_weights.get(row['position_group'])
    if weights is None:
        return None  # no weights defined for this position group
    score = 0
    for metric, weight in weights.items():
        score += row[metric] * weight
    return score

In [40]:
df_outfield['performance_score'] = df_outfield.apply(
    calculate_weighted_score,
    axis=1
)
df_outfield.head()

,player_id,first_name,last_name,player_name,current_club_id,country_of_birth,city_of_birth,country_of_citizenship,date_of_birth,sub_position,...,xG_per90,xAG_per90,progressive_passes_per90,progressive_carries_per90,dribbles_successful_per90,shot_creating_actions_per90,goal_creating_actions_per90,tackles_per90,blocks_per90,performance_score
1,29091,NaN,Nenê,Nenê,10492,Brazil,Jundiaí,Brazil,1981-07-19 00:00:00,Attacking Midfield,...,0.095618,0.256972,4.362550,1.374502,0.717131,4.482072,0.239044,0.597610,0.478088,14.061753
2,29241,Thiago,Silva,Thiago Silva,2462,Brazil,Rio de Janeiro,Brazil,1984-09-22 00:00:00,Centre-Back,...,0.035714,0.050000,2.428571,0.142857,0.000000,0.928571,0.000000,0.928571,0.714286,9.500000
3,37647,Dimitri,Payet,Dimitri Payet,978,Réunion,Saint-Pierre,France,1987-03-29 00:00:00,Attacking Midfield,...,0.125410,0.184426,7.450820,1.770492,1.254098,4.500000,0.663934,0.885246,0.147541,15.347951
4,45319,Gabriel,Mercado,Gabriel Mercado,6600,Argentina,Puerto Madryn,Argentina,1987-03-18 00:00:00,Centre-Back,...,0.022766,0.045531,2.428331,0.151771,0.075885,0.682968,0.151771,1.290051,0.531197,9.865093
5,46741,NaN,David Luiz,David Luiz,10870,Brazil,Diadema,Brazil,1987-04-22 00:00:00,Centre-Back,...,0.097380,0.020501,5.330296,1.640091,0.205011,1.588838,0.051253,0.563781,0.973804,18.041002


## Verifying weighted performance scores

In [41]:
df_outfield[['player_name', 'position_group', 'performance_score']].head(10)

,player_name,position_group,performance_score
1,Nenê,Attacking Midfielder,14.061753
2,Thiago Silva,Centre-Back,9.500000
3,Dimitri Payet,Attacking Midfielder,15.347951
4,Gabriel Mercado,Centre-Back,9.865093
5,David Luiz,Centre-Back,18.041002
6,Fagner,Full-Back,18.621030
9,Willian,Forward,3.820498
10,Everton Ribeiro,Attacking Midfielder,13.014920
14,Germán Cano,Forward,2.911765
15,Titi,Centre-Back,10.581683


In [42]:
df_outfield['performance_score'].describe()

count    182.000000
mean      15.619510
std        7.843357
min        2.911765
25%        8.916898
50%       15.516286
75%       21.068316
max       44.661386
Name: performance_score, dtype: float64

In [58]:
df_outfield[(df_outfield['position_group'] == 'Centre-Back') & ((df_outfield['player_name'] == 'Thiago Silva') | (df_outfield['player_name'] == 'David Luiz')) ].nlargest(10, 'performance_score')[['player_name', 'performance_score', 'minutes_played', 'tackles', 'blocks', 'progressive_passes', 'progressive_carries']]

,player_name,performance_score,minutes_played,tackles,blocks,progressive_passes,progressive_carries
5,David Luiz,18.041002,1756.0,11.0,19.0,104.0,32.0
2,Thiago Silva,9.500000,1260.0,13.0,10.0,34.0,2.0
